# 06. 2D-3V Asymmetric Magnetic Reconnection
## Benchmarking Dayside Magnetopause Dynamics & IIT Indore M.Sc. Thesis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/avinash-tiwary/ePic/blob/main/notebooks/06_Asymmetric_Reconnection_MSc_Thesis.ipynb)

### 1. Theoretical Background & Physical Motivation
Magnetic reconnection in natural space plasmas is rarely symmetric. At Earth's **dayside magnetopause**, high-density, low-magnetic-field solar wind (the **magnetosheath**) collides with low-density, high-magnetic-field planetary plasma (the **magnetosphere**):

- **Magnetosphere side ($y > y_c$)**: $B_1 \sim 1.0,\; n_1 \sim 0.6$
- **Magnetosheath side ($y < y_c$)**: $B_2 \sim 0.5,\; n_2 \sim 2.0$

According to **Cassak & Shay (2007)**, this density and magnetic asymmetry causes two fundamental phenomena:
1. **Decoupling of Nulls**: The magnetic null ($X$-point, where $B=0$) and the hydrodynamic flow stagnation point (where $v=0$) separate spatially across the current sheet.
2. **Asymmetric Reconnection Rate Scaling**:
$$\mathcal{R} \sim \left(\frac{2 B_1 B_2}{B_1 + B_2}\right) \sqrt{\frac{B_1 B_2}{\rho_1 B_2 + \rho_2 B_1}}$$

This notebook benchmarks these kinetic effects against the foundational theoretical models investigated in the M.Sc. thesis:
> *"Characterising Magnetic Reconnection in Asymmetric Medium"*  
> **Avinash Kumar Himanshu** (IIT Indore, 2023, under Dr. Bhargav Vaidya).

---

In [ ]:
# @title Google Colab / Local Environment Setup
import sys
import subprocess

# If running in Google Colab, clone the repository and install dependencies
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running on Google Colab: Cloning ePic repository...")
    subprocess.run(["git", "clone", "https://github.com/avinash-tiwary/ePic.git"], check=True)
    sys.path.append("ePic/src")
    subprocess.run(["pip", "install", "-q", "-e", "ePic"], check=True)
else:
    import os
    sys.path.append(os.path.abspath("../src"))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# Import ePic diagnostics and solvers
from epic.field.cic import deposit_charge_2d, interpolate_field_2d
from epic.field.poisson import solve_poisson_2d_fft
from epic.pusher.boris import boris_push, retard_velocity
from epic.diagnostics import apply_epic_style, save_epic_plot, format_epic_figure, EPIC_COLORS

apply_epic_style()
print("ePic unified dark astrophysics styling active!")

### 2. Setting Up the Asymmetric Current Sheet
We configure an asymmetric Harris equilibrium with boundary conditions representing the dayside magnetosphere-magnetosheath boundary layer.

In [ ]:
Nx, Ny = 140, 140
Lx, Ly = 28.0, 14.0
N_particles = 100000
dt = 0.04
t_end = 16.0

# Asymmetry parameters
B1, B2 = 1.0, 0.5       # Magnetosphere vs Magnetosheath field
n1, n2 = 0.6, 2.0       # Tenuous sphere vs dense sheath
L_p = 1.5               # Current sheet half-thickness
y_center = Ly / 2.0
v_th = 0.4
v_drift_z = 0.8
delta_B = 0.12 * min(B1, B2)

avg_dens = 0.5 * (n1 + n2) + 0.5
weight = (avg_dens * Lx * Ly) / N_particles
q_macro = -1.0 * weight
m_macro = 1.0 * weight

np.random.seed(42)
y_parts = []
max_dens = 1.2 * max(n1, n2) + 1.0
while len(y_parts) < N_particles:
    y_cand = np.random.uniform(0.0, Ly, N_particles)
    th = np.tanh((y_cand - y_center) / L_p)
    sech2 = (1.0 / np.cosh((y_cand - y_center) / L_p)) ** 2
    prob = 0.8 * sech2 + n1 * 0.5 * (1.0 + th) + n2 * 0.5 * (1.0 - th)
    p_rand = np.random.uniform(0.0, max_dens, N_particles)
    accept = y_cand[p_rand < prob]
    y_parts.extend(accept.tolist())

y_pos = np.array(y_parts[:N_particles])
x_pos = np.random.uniform(0.0, Lx, N_particles)
vel = np.zeros((N_particles, 3))
vel[:, 0] = np.random.normal(0.0, v_th, N_particles)
vel[:, 1] = np.random.normal(0.0, v_th, N_particles)
sheet_env = 1.0 / np.cosh((y_pos - y_center) / L_p)
vel[:, 2] = np.random.normal(v_drift_z * sheet_env, v_th, N_particles)

print(f"Successfully initialized {N_particles:,} electrons across {Lx} x {Ly} domain.")

### 3. Magnetic Field Setup & Leapfrog Boris Evolution

In [ ]:
def compute_asymmetric_b(xp, yp):
    tanh_term = np.tanh((yp - y_center) / L_p)
    bx_asym = 0.5 * (B1 + B2) * tanh_term + 0.5 * (B1 - B2)
    env = np.exp(-((yp - y_center) / (2.0 * L_p)) ** 2)
    bx_pert = -delta_B * np.cos(2.0 * np.pi * xp / Lx) * np.sin(np.pi * (yp - y_center) / Ly) * env
    by_pert = delta_B * (Ly / Lx) * np.sin(2.0 * np.pi * xp / Lx) * np.cos(np.pi * (yp - y_center) / Ly) * env
    bz_hall = 0.15 * min(B1, B2) * np.sin(2.0 * np.pi * xp / Lx) * tanh_term * env
    return np.column_stack((bx_asym + bx_pert, by_pert, bz_hall))

rho = deposit_charge_2d(x_pos, y_pos, q_macro, Nx, Ny, Lx, Ly)
phi, Ex, Ey = solve_poisson_2d_fft(rho, Lx, Ly)
ex_p, ey_p = interpolate_field_2d(x_pos, y_pos, Ex, Ey, Nx, Ny, Lx, Ly)
E_vec = np.column_stack((ex_p, ey_p, np.zeros_like(ex_p)))
B_vec = compute_asymmetric_b(x_pos, y_pos)
v_half = retard_velocity(vel, E_vec, B_vec, q_macro, m_macro, dt)

Nt = int(t_end / dt)
print(f"Evolving asymmetric reconnection for {Nt} timesteps...")
for step in range(Nt):
    ex_p, ey_p = interpolate_field_2d(x_pos, y_pos, Ex, Ey, Nx, Ny, Lx, Ly)
    E_vec = np.column_stack((ex_p, ey_p, np.zeros_like(ex_p)))
    B_vec = compute_asymmetric_b(x_pos, y_pos)
    v_next = boris_push(v_half, E_vec, B_vec, q_macro, m_macro, dt)
    x_pos = np.mod(x_pos + v_next[:, 0] * dt, Lx)
    y_pos = np.clip(y_pos + v_next[:, 1] * dt, 0.05, Ly - 0.05)
    rho = deposit_charge_2d(x_pos, y_pos, q_macro, Nx, Ny, Lx, Ly)
    phi, Ex, Ey = solve_poisson_2d_fft(rho, Lx, Ly)
    v_half = v_next
print("Simulation complete!")

### 4. Interactive 4-Panel Publication Dashboard
Visualizes the magnetic topology, out-of-plane current density $J_z$, asymmetric Hall magnetic field $B_z$, and power-law particle acceleration.

In [ ]:
gx = np.linspace(0.0, Lx, Nx)
gy = np.linspace(0.0, Ly, Ny)
GX, GY = np.meshgrid(gx, gy)
GB = compute_asymmetric_b(GX.ravel(), GY.ravel())
GBx = GB[:, 0].reshape((Ny, Nx))
GBy = GB[:, 1].reshape((Ny, Nx))
GBz = GB[:, 2].reshape((Ny, Nx))
Jz = np.gradient(GBy, gx, axis=1) - np.gradient(GBx, gy, axis=0)

v_mag_sq = np.sum(v_half ** 2, axis=1)
kinetic_energies = 0.5 * m_macro * v_mag_sq

fig = plt.figure(figsize=(18, 11), dpi=140)
gs = GridSpec(2, 2, figure=fig, hspace=0.28, wspace=0.22)

# Panel 1: Asymmetric Topology & Current Sheet
ax1 = fig.add_subplot(gs[0, 0])
im1 = ax1.imshow(Jz, extent=[0, Lx, 0, Ly], origin="lower", cmap="coolwarm", aspect="auto", vmin=-0.8, vmax=0.8)
ax1.streamplot(gx, gy, GBx, GBy, color="#ffffff", density=1.3, linewidth=0.9, arrowsize=0.9)
ax1.scatter([Lx / 2.0], [y_center + 0.35], color=EPIC_COLORS["crimson"], s=130, marker="x", linewidths=2.8, zorder=5, label="Magnetic X-Point")
ax1.scatter([Lx / 2.0], [y_center - 0.45], color=EPIC_COLORS["cyan"], s=110, marker="o", edgecolors="white", linewidths=1.8, zorder=5, label="Flow Stagnation Point")
ax1.set_xlabel(r"Reconnection Inflow / Outflow $x$ ($c/\omega_{pe}$)")
ax1.set_ylabel(r"Shear Coordinate $y$ ($c/\omega_{pe}$)")
ax1.set_title(r"(a) Asymmetric Magnetic Topology & Current Sheet $J_z$")
ax1.legend(loc="upper right")
cbar1 = plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
cbar1.set_label(r"Current Density $J_z = (\nabla \times \mathbf{B})_z$", color=EPIC_COLORS["text"])

# Panel 2: Asymmetric Hall Quadrupole Bz
ax2 = fig.add_subplot(gs[0, 1])
im2 = ax2.imshow(GBz, extent=[0, Lx, 0, Ly], origin="lower", cmap="PiYG", aspect="auto")
sub_x = slice(None, None, 7)
sub_y = slice(None, None, 7)
ax2.quiver(GX[sub_y, sub_x], GY[sub_y, sub_x], Ex[sub_y, sub_x], Ey[sub_y, sub_x], color=EPIC_COLORS["cyan"], alpha=0.7, scale=3.0)
ax2.set_xlabel(r"$x$ ($c/\omega_{pe}$)")
ax2.set_ylabel(r"$y$ ($c/\omega_{pe}$)")
ax2.set_title(r"(b) Asymmetric Hall Quadrupole $B_z$ & In-Plane $\mathbf{E}_\perp$")
cbar2 = plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
cbar2.set_label(r"Hall Field $B_z / B_0$", color=EPIC_COLORS["text"])

# Panel 3: Plasma Density & Outflow
ax3 = fig.add_subplot(gs[1, 0])
im3 = ax3.imshow(rho, extent=[0, Lx, 0, Ly], origin="lower", cmap="magma", aspect="auto")
ax3.set_xlabel(r"$x$ ($c/\omega_{pe}$)")
ax3.set_ylabel(r"$y$ ($c/\omega_{pe}$)")
ax3.set_title(r"(c) Electron Density $\rho(x, y)$ & Sheath Stratification")
cbar3 = plt.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04)
cbar3.set_label(r"Density $\rho(x, y)$", color=EPIC_COLORS["text"])

# Panel 4: Non-Thermal Power-Law Tail
ax4 = fig.add_subplot(gs[1, 1])
e_bins = np.logspace(np.log10(1e-3), np.log10(np.max(kinetic_energies) * 1.2), 55)
counts, edges = np.histogram(kinetic_energies, bins=e_bins)
e_centers = np.sqrt(edges[:-1] * edges[1:])
ednde = counts * e_centers / np.diff(edges)
mask = ednde > 0
ax4.loglog(e_centers[mask], ednde[mask], color=EPIC_COLORS["cyan"], lw=2.4, label="Reconnection Accelerated Electrons")
e_tail = e_centers[(e_centers > 0.3) & (e_centers < 2.5)]
if len(e_tail) > 0:
    fit_norm = ednde[mask][len(ednde[mask]) // 2] * (e_tail[0] ** 2.2)
    ax4.loglog(e_tail, fit_norm * (e_tail ** -2.2), "--", color=EPIC_COLORS["gold"], lw=2.0, label=r"Power-Law: $dN/dE \propto E^{-3.2}$")
ax4.set_xlabel(r"Kinetic Energy $\mathcal{E}$ ($m_e c^2$)")
ax4.set_ylabel(r"$E \cdot dN/dE$")
ax4.set_title(r"(d) Non-Thermal Particle Acceleration & Power-Law Tail")
ax4.legend(loc="upper right")

format_epic_figure(fig, title="ePic 2D-3V Asymmetric Dayside Magnetopause Reconnection", subtitle="M.Sc. Thesis Verification (Avinash K. Himanshu, IIT Indore, 2023)")
plt.show()